# Numerikus módszerek – 9. hét előadás
## Iteratív LER-módszerek · Nemlineáris egyenletek - Interpoláció

**Tartalom:**
1. Gauss–Seidel és Relaxált Gauss–Seidel (SOR) iteráció
2. Richardson-iteráció
3. Bolzano-tétel, intervallumfelezés
4. Fixponttételek, egyszerű iterációk
5. Konvergencia rend
6. Newton-módszer
7. Húrmódszer és szelőmódszer
8. Többváltozós Newton-módszer
9. Interpoláció – Lagrange-alak
10. Hibaformula
11. Newton-alak (osztott differenciák)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import *
np.set_printoptions(precision=6, suppress=True)
plt.rcParams['figure.figsize'] = (8, 4)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "d:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "d:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed

---
## 1. Gauss–Seidel és Relaxált Gauss–Seidel (SOR) iteráció

### Elmélet

Legyen $Ax = b$, ahol $A = D - L - U$ ($D$: diagonális, $L$: alsó, $U$: felső háromszög).

**Gauss–Seidel (GS) iteráció $S(1)$:**
$$x^{(k+1)} = (D-L)^{-1} U\, x^{(k)} + (D-L)^{-1} b, \quad B_S = (D-L)^{-1}U$$

**Relaxált GS (SOR) $S(\omega)$:**
$$x^{(k+1)} = \left(D - \omega L\right)^{-1}\!\left[(1-\omega)D + \omega U\right] x^{(k)}
+ \omega\left(D-\omega L\right)^{-1} b$$

> **Tétel** – szimmetrikus, pozitív definit, *tridiagonális* mátrixra:
> $$\omega_0 = \frac{2}{1+\sqrt{1-\varrho(B_J)^2}}, \qquad
> \varrho\!\left(B_{S(\omega_0)}\right) = \omega_0 - 1.$$

### Kidolgozott példa

$$A = \begin{bmatrix}2&-1&0\\-1&2&-1\\0&-1&2\end{bmatrix}, \quad b = \begin{bmatrix}1\\0\\0\end{bmatrix}$$

$B_J$ sajátértékei: $0,\;\pm\tfrac{1}{\sqrt{2}}$, tehát $\varrho(B_J)=\tfrac{1}{\sqrt{2}}$.

$$\omega_0 = \frac{2}{1+\sqrt{1-\tfrac{1}{2}}} = \frac{2}{1+\tfrac{1}{\sqrt{2}}} = 4-2\sqrt{2}\approx 1.1716$$
$$\varrho\!\left(B_{S(\omega_0)}\right) = \omega_0 - 1 = 3-2\sqrt{2}\approx 0.1716$$

In [ ]:
def sor_iter(A, b, x0, omega, n_iter):
    """SOR iteráció. omega=1 -> Gauss-Seidel."""
    n = len(b)
    x = x0.copy().astype(float)
    hist = [x.copy()]
    for _ in range(n_iter):
        x_new = x.copy()
        for i in range(n):
            sigma = sum(A[i,j]*x_new[j] for j in range(i)) \
                  + sum(A[i,j]*x[j]    for j in range(i+1, n))
            x_new[i] = (1-omega)*x[i] + omega*(b[i] - sigma)/A[i,i]
        x = x_new
        hist.append(x.copy())
    return np.array(hist)

A = np.array([[2,-1,0],[-1,2,-1],[0,-1,2]], float)
b = np.array([1,0,0], float)
x_exact = np.linalg.solve(A, b)
print('Pontos megoldás:', x_exact)

rho_BJ = 1/np.sqrt(2)
omega0 = 2 / (1 + np.sqrt(1 - rho_BJ**2))
print(f'rho(B_J) = {rho_BJ:.6f}')
print(f'omega_0  = {omega0:.6f}  (=4-2sqrt(2)={4-2*np.sqrt(2):.6f})')
print(f'rho(B_S(omega0)) = {omega0-1:.6f}')

In [ ]:
# Tapasztalati kontrakciós együttható vizsgálata különböző omega értékekre
A2 = np.array([[4,-1,0],[-1,4,-1],[0,-1,4]], float)
b2 = np.array([3,2,3], float)
x_sol = np.linalg.solve(A2, b2)  # = [1,1,1]
print('Pontos megoldás:', x_sol)

omegas = [1.0, 0.8, 0.6, 1.033, -0.1, 2.0, 2.5]
x0 = np.zeros(3)
N = 30

fig, axes = plt.subplots(2, 4, figsize=(14,6))
axes = axes.flatten()
for idx, om in enumerate(omegas):
    hist = sor_iter(A2, b2, x0, om, N)
    errs = np.linalg.norm(hist - x_sol, axis=1)
    # tapasztalati kontrakciós együttható
    q_emp = np.where(errs[:-1]>1e-15, errs[1:]/errs[:-1], np.nan)
    ax = axes[idx]
    ax.scatter(range(1,N+1), q_emp, s=20, color='red')
    q_lim = np.nanmean(q_emp[-10:]) if not np.all(np.isnan(q_emp)) else float('inf')
    diverge = q_lim > 1.0
    title = f'omega={om}' + (' [DIVERGENS]' if diverge else f'  q≈{q_lim:.4f}')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('k'); ax.set_ylabel('q_k')
axes[-1].axis('off')
plt.suptitle('S(ω) iteráció tapasztalati kontrakciós együtthatói', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. Richardson-iteráció

### Elmélet

**Definíció** – Richardson-iteráció $p$ paraméterrel – $R(p)$:
$$x^{(k+1)} = \underbrace{(I-pA)}_{B_{R(p)}} x^{(k)} + \underbrace{pb}_{c_{R(p)}}$$

**Reziduum-vektoros alak** ($r^{(k)} = b - Ax^{(k)}$, $s^{(k)} = p\,r^{(k)}$):
$$x^{(k+1)} = x^{(k)} + s^{(k)}, \qquad r^{(k+1)} = r^{(k)} - A\,s^{(k)}$$

> **Tétel** – Ha $A\in\mathbb{R}^{n\times n}$ szimmetrikus, pozitív definit,
> sajátértékei $m=\lambda_1\le\cdots\le\lambda_n=M$, akkor $R(p)$ konvergens
> $\Leftrightarrow$ $p\in\left(0,\,\tfrac{2}{M}\right)$.
> Optimális paraméter és kontrakciós együttható:
> $$p_0 = \frac{2}{M+m}, \qquad q = \frac{M-m}{M+m}.$$

### Kidolgozott példa

$$A = \begin{bmatrix}3&-1\\-1&3\end{bmatrix},\quad b=\begin{bmatrix}1\\5\end{bmatrix}$$

Sajátértékek: $m=2$, $M=4$.
$$p_0 = \frac{2}{4+2}=\frac{1}{3},\quad q=\frac{4-2}{4+2}=\frac{1}{3}\approx 0.333$$

In [ ]:
def richardson(A, b, x0, p, n_iter):
    x = x0.copy().astype(float)
    r = b - A @ x
    hist = [x.copy()]
    for _ in range(n_iter):
        s = p * r
        x = x + s
        r = r - A @ s
        hist.append(x.copy())
    return np.array(hist)

A = np.array([[3,-1],[-1,3]], float)
b = np.array([1,5], float)
x_exact = np.linalg.solve(A, b)
print('Pontos megoldás:', x_exact)

eigvals = np.linalg.eigvalsh(A)
m, M = eigvals.min(), eigvals.max()
p0 = 2/(M+m)
q  = (M-m)/(M+m)
print(f'm={m}, M={M}, p0={p0:.4f}, q={q:.4f}')

hist = richardson(A, b, np.zeros(2), p0, 30)
errs = np.linalg.norm(hist - x_exact, axis=1)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.semilogy(errs, 'b-o', markersize=4)
plt.title(f'Richardson hiba (p=p₀={p0:.3f})')
plt.xlabel('iteráció'); plt.ylabel('||hiba||')
plt.grid(True)

# rho(B_R(p)) függvény p-ben
plt.subplot(1,2,2)
ps = np.linspace(0, 0.6, 500)
rho_p = np.array([max(abs(1-p*m), abs(1-p*M)) for p in ps])
plt.plot(ps, rho_p, 'b-', label=r'$\varrho(B_{R(p)})$')
plt.axvline(p0, color='g', linestyle='--', label=f'p₀={p0:.3f}')
plt.axhline(q, color='r', linestyle=':', label=f'q={q:.3f}')
plt.axvline(2/M, color='orange', linestyle=':', label=f'2/M={2/M:.3f}')
plt.ylim(0, 1.2); plt.legend(); plt.grid(True)
plt.title(r'$\varrho(B_{R(p)})$ függvény')
plt.tight_layout(); plt.show()

---
## 3. Bolzano-tétel és intervallumfelezés

### Elmélet

> **Bolzano-tétel:** Ha $f\in C[a,b]$ és $f(a)\cdot f(b)<0$, akkor
> $\exists\,x^*\in(a,b): f(x^*)=0$.

**Intervallumfelezés algoritmus:**
Minden lépésben $c = (a+b)/2$. Ha $f(a)\cdot f(c)<0$: új $b\leftarrow c$, egyébként $a\leftarrow c$.

**Hibabecslés:** $|x_k - x^*| \le \dfrac{b-a}{2^k}$

### Kidolgozott példa

$f(x) = x^3 - x - 1 = 0$ gyöke $[1,2]$-n.
$f(1)=-1<0$, $f(2)=5>0$ ✓

Szükséges lépésszám $10^{-6}$ pontossághoz:
$$\frac{b-a}{2^k} < 10^{-6} \Rightarrow k > \frac{\ln(b-a)-\ln(10^{-6})}{\ln 2} = \frac{\ln 10^6}{\ln 2} \approx 19.9$$

In [ ]:
def bisect(f, a, b, tol=1e-10, max_iter=100):
    hist = []
    for k in range(max_iter):
        c = (a+b)/2
        hist.append({'k':k, 'a':a, 'b':b, 'c':c, 'f(c)':f(c), 'err_bound':(b-a)/2})
        if abs(f(c)) < tol or (b-a)/2 < tol:
            break
        if f(a)*f(c) < 0:
            b = c
        else:
            a = c
    return c, hist

f = lambda x: x**3 - x - 1
root, hist = bisect(f, 1, 2, tol=1e-10)
print(f'Gyök: x* ≈ {root:.10f}')
print(f'Ellenőrzés: f(x*) = {f(root):.2e}')
print(f'Szükséges lépésszám 1e-6 pontossághoz: {np.log(1e6)/np.log(2):.1f}')
print()
print(f'{'k':>3}  {'a':>12}  {'b':>12}  {'c':>12}  {'f(c)':>12}  {'hiba korlát':>12}')
for r in hist[:10]:
    print(f"{r['k']:>3}  {r['a']:>12.8f}  {r['b']:>12.8f}  {r['c']:>12.8f}  {r['f(c)']:>12.6f}  {r['err_bound']:>12.8f}")

# Ábra
xs = np.linspace(0.5, 2.5, 300)
plt.figure(figsize=(7,4))
plt.plot(xs, f(xs), 'b-', label='$f(x)=x^3-x-1$')
plt.axhline(0, color='k', linewidth=0.8)
plt.axvline(root, color='r', linestyle='--', label=f'x*≈{root:.4f}')
plt.scatter([r['c'] for r in hist[:8]], [f(r['c']) for r in hist[:8]],
            c=range(8), cmap='autumn', zorder=5, label='felezési pontok')
plt.legend(); plt.grid(True); plt.title('Intervallumfelezés'); plt.show()

---
## 4. Fixponttételek és egyszerű iterációk

### Banach-féle fixponttétel

Ha $\varphi:[a,b]\to[a,b]$ kontrakció ($|\varphi(x)-\varphi(y)|\le q|x-y|$, $q<1$), akkor:
1. $\exists!\;x^*\in[a,b]:\;x^*=\varphi(x^*)$
2. $\forall x_0$: $x_{k+1}=\varphi(x_k)\to x^*$
3. Hibabecslések: $|x_k-x^*|\le q^k|x_0-x^*|$ és $|x_k-x^*|\le\dfrac{q^k}{1-q}|x_1-x_0|$

### Kidolgozott példa 1 – $x = \cos(x)$

$\varphi(x)=\cos(x)$ a $[0,1]$ intervallumon:
- $\varphi([0,1])=[\cos(1),1]\subset[0,1]$ ✓
- $|\varphi'(x)|=|{-\sin(x)}|\le\sin(1)\approx 0.8415=q<1$ ✓

Hibabecslés $10^{-1}$ pontossághoz: $0.8415^k < 0.1 \Rightarrow k > 13.34$, tehát **14 lépés**.

### Kidolgozott példa 2 – $x^3 - x - 1 = 0$

(a) $\varphi(x)=x^3-1$: $|\varphi'(x^*)| = 3(x^*)^2 \approx 3\cdot 1.3247^2 \approx 5.27 > 1$ → **divergens**

(b) $\varphi(x)=\sqrt[3]{x+1}$: $|\varphi'(x^*)| = \tfrac{1}{3}(x^*+1)^{-2/3}\approx 0.26 < 1$ → **konvergens**

In [ ]:
# x = cos(x) iteráció
phi = np.cos
x = 0.5
iterates = [x]
for _ in range(50):
    x = phi(x)
    iterates.append(x)

x_star = iterates[-1]
errs = np.abs(np.array(iterates) - x_star)

# tapasztalati q
q_emp = errs[1:] / np.where(errs[:-1]>1e-15, errs[:-1], np.nan)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,4))
ax1.plot(iterates[:30], 'b-o', markersize=4)
ax1.axhline(x_star, color='r', linestyle='--', label=f'x*≈{x_star:.6f}')
ax1.set_title('x = cos(x) iteráció'); ax1.legend(); ax1.grid(True)

ax2.scatter(range(1,len(q_emp)+1), q_emp, s=15, color='red')
ax2.axhline(np.sin(1), color='g', linestyle='--', label=f'sin(1)≈{np.sin(1):.4f}')
ax2.set_ylim(0,1.1); ax2.set_title('Tapasztalati q értékek')
ax2.legend(); ax2.grid(True)
plt.suptitle('x = cos(x) fixpont-iteráció'); plt.tight_layout(); plt.show()
print(f'Fixpont: x* ≈ {x_star:.8f},  ellenőrzés: cos(x*)={np.cos(x_star):.8f}')
print(f'Szükséges lépésszám 0.1 pontossághoz: k > {-np.log(0.1)/np.log(1/np.sin(1)):.2f}')

In [ ]:
# x^3 - x - 1 = 0: divergens vs konvergens iteráció
phi_div = lambda x: x**3 - 1
phi_con = lambda x: (x+1)**(1/3)
x_star_true = 1.3247179572  # közelítő

def run_iter(phi, x0, n, x_star):
    x = x0; hist = [x]
    for _ in range(n):
        xn = phi(x)
        if abs(xn) > 1e6: break
        x = xn; hist.append(x)
    return hist

h_div = run_iter(phi_div, 1.5, 6, x_star_true)
h_con = run_iter(phi_con, 1.5, 20, x_star_true)

fig, (a1, a2) = plt.subplots(1,2,figsize=(11,4))
a1.plot(h_div, 'r-o'); a1.set_title('φ(x)=x³-1 (DIVERGENS)'); a1.grid(True)
a2.plot(h_con, 'g-o'); a2.axhline(x_star_true, color='r', ls='--', label=f'x*≈{x_star_true:.4f}')
a2.set_title('φ(x)=∛(x+1) (konvergens)'); a2.legend(); a2.grid(True)
plt.tight_layout(); plt.show()

---
## 5. Konvergencia rend

### Definíció

Az $(x_k)$ sorozat $x^*$-hoz **$p$-edrendben konvergál**, ha
$$\lim_{k\to\infty}\frac{|x_{k+1}-x^*|}{|x_k-x^*|^p} = c \in (0,+\infty).$$

- $p=1$: **elsőrendű** (lineáris) konvergencia ($c\le 1$)
- $p=2$: **másodrendű** (kvadratikus) konvergencia
- $p>1$: szuperlineáris

> **Tétel** – Ha $\varphi\in C^p[a,b]$, $x_{k+1}=\varphi(x_k)\to x^*$,
> és $\varphi'(x^*)=\cdots=\varphi^{(p-1)}(x^*)=0$, de $\varphi^{(p)}(x^*)\ne 0$,
> akkor a konvergencia $p$-edrendű és
> $$|x_{k+1}-x^*| \le \frac{M_p}{p!}|x_k-x^*|^p.$$

### √2 közelítése különböző rendű iterációkkal (ea_09 MATLAB-példa)

| Iteráció | Rend |
|----------|------|
| $x_{k+1}=1+\tfrac{1}{1+x_k}$ (lánctört) | $p=1$ |
| $x_{k+1}=\tfrac{1}{2}\!\left(x_k+\tfrac{2}{x_k}\right)$ (Newton) | $p=2$ |
| $x_{k+1}=x_k\cdot\tfrac{x_k^2+6}{3x_k^2+2}$ (Taylor) | $p=3$ |

In [ ]:
sqrt2 = np.sqrt(2)

phi1 = lambda x: 1 + 1/(1+x)      # p=1
phi2 = lambda x: 0.5*(x + 2/x)    # p=2  (Newton f(x)=x^2-2)
phi3 = lambda x: x*(x**2+6)/(3*x**2+2)  # p=3

x0 = 1.0
N = 12
errs = {1:[], 2:[], 3:[]}
phis = {1:phi1, 2:phi2, 3:phi3}
for p, phi in phis.items():
    x = x0
    for _ in range(N):
        x = phi(x)
        errs[p].append(abs(x - sqrt2))

plt.figure(figsize=(9,4))
for p, e in errs.items():
    valid = [(i,v) for i,v in enumerate(e) if v > 1e-16]
    if valid:
        xs_, ys_ = zip(*valid)
        plt.semilogy(xs_, ys_, 'o-', label=f'p={p}')
plt.xlabel('iteráció'); plt.ylabel('|x_k - √2|')
plt.title('√2 közelítése különböző rendű iterációkkal')
plt.legend(); plt.grid(True); plt.show()

print('Utolsó hibák:')
for p, e in errs.items():
    print(f'  p={p}: {e[-1]:.2e}')

---
## 6. Newton-módszer

### Elmélet

$$x^{(k+1)} = x^{(k)} - \frac{f(x^{(k)})}{f'(x^{(k)})}$$

Geometriai értelmezés: az $f$ érintőjének nullhelye.

> **Tétel** – Ha $f\in C^2[a,b]$, $\exists\,x^*\in(a,b):\,f(x^*)=0$, $f'(x^*)\ne 0$,
> és $x_0$ elég közel van $x^*$-hoz, akkor a Newton-módszer **másodrendben** konvergál:
> $$|x_{k+1}-x^*| \le \frac{M_2}{2m_1}|x_k-x^*|^2, \quad
> M_2=\max|f''|,\;m_1=\min|f'|.$$

### Kidolgozott példa: $f(x)=x^2-2$, tehát $x^*=\sqrt{2}$

$$x_{k+1} = x_k - \frac{x_k^2-2}{2x_k} = \frac{1}{2}\!\left(x_k+\frac{2}{x_k}\right)$$

In [ ]:
def newton(f, df, x0, tol=1e-12, max_iter=50):
    x = float(x0); hist = [x]
    for _ in range(max_iter):
        dx = f(x)/df(x)
        x -= dx
        hist.append(x)
        if abs(dx) < tol: break
    return x, hist

f  = lambda x: x**2 - 2
df = lambda x: 2*x
root, hist = newton(f, df, 1.0)
errs = np.abs(np.array(hist) - np.sqrt(2))

print('Newton-lépések (f(x)=x²-2):')
print(f'{'k':>3}  {'x_k':>18}  {'hiba':>12}')
for k, (xk, ek) in enumerate(zip(hist, errs)):
    print(f'{k:>3}  {xk:>18.14f}  {ek:>12.6e}')

# log-log ábra: másodrendű lejtő
valid = [(i,e) for i,e in enumerate(errs[1:],1) if e>1e-16 and errs[i-1]>1e-16]
if len(valid)>=2:
    ki, ei = zip(*valid)
    plt.figure(figsize=(7,4))
    plt.loglog(errs[:-1][list(np.array(ki)-1)], ei, 'b-o', label='Newton')
    # p=2 referencia vonal
    e0 = errs[1]
    ref_x = np.array([errs[1], errs[2]])
    plt.loglog(ref_x, ref_x**2/e0, 'r--', label='p=2 referencia')
    plt.xlabel('|x_k - x*|'); plt.ylabel('|x_{k+1} - x*|')
    plt.title('Newton: másodrendű konvergencia'); plt.legend(); plt.grid(True); plt.show()

---
## 7. Húrmódszer és szelőmódszer

### Húrmódszer (regula falsi)

$$x_{k+1} = x_k - f(x_k)\cdot\frac{b-a}{f(b)-f(a)}$$

(az $a$, $b$ végpontok **rögzítve** maradnak, ahol $f(a)\cdot f(b)<0$)

> **Tétel** – feltételek mellett elsőrendben konvergál.
> Ha $f\in C^2[a,b]$, $f'$ állandó előjelű, $|x_0-x^*|$, $|x_1-x^*| < r$, akkor
> $p = \frac{1+\sqrt{5}}{2}$-rendben konvergál. **Biz.:** nélkül.

### Szelőmódszer

$$x_{k+1} = x_k - f(x_k)\cdot\frac{x_k-x_{k-1}}{f(x_k)-f(x_{k-1})}$$

Két kezdőpont kell: $x_0, x_1\in[a,b]$. Konvergencia rend: $p=\dfrac{1+\sqrt{5}}{2}\approx 1.618$.

### Összehasonlítás: $f(x)=x^3-x-1=0$

In [ ]:
f  = lambda x: x**3 - x - 1
df = lambda x: 3*x**2 - 1
a, b_ = 1.0, 2.0
x_star = 1.3247179572

# Intervallumfelezés
_, bist = bisect(f, a, b_, tol=1e-12)
err_bis = [abs(r['c']-x_star) for r in bist]

# Húrmódszer
def chord(f, a, b, n_iter):
    hist = []
    fa, fb = f(a), f(b)
    x = a
    for _ in range(n_iter):
        x = x - f(x)*(b-a)/(fb-fa)
        hist.append(x)
    return hist

# Szelőmódszer
def secant(f, x0, x1, n_iter):
    hist = [x0, x1]
    for _ in range(n_iter):
        fx0, fx1 = f(hist[-2]), f(hist[-1])
        if abs(fx1-fx0) < 1e-15: break
        xn = hist[-1] - fx1*(hist[-1]-hist[-2])/(fx1-fx0)
        hist.append(xn)
    return hist[2:]

# Newton
_, newt_h = newton(f, df, 1.5)
err_newt = [abs(x-x_star) for x in newt_h[1:]]

chord_h = chord(f, a, b_, 30)
err_chord = [abs(x-x_star) for x in chord_h]

sec_h = secant(f, 1.0, 2.0, 20)
err_sec = [abs(x-x_star) for x in sec_h]

plt.figure(figsize=(8,5))
def plot_err(errs, label):
    e = [v for v in errs if v>1e-16]
    plt.semilogy(range(len(e)), e, '-o', markersize=4, label=label)

plot_err(err_bis[:25],   'Intervallumfelezés (p=1)')
plot_err(err_chord[:25], 'Húrmódszer (p=1)')
plot_err(err_sec[:15],   'Szelőmódszer (p≈1.618)')
plot_err(err_newt[:10],  'Newton (p=2)')
plt.xlabel('iteráció'); plt.ylabel('|x_k - x*|')
plt.title('Módszerek összehasonlítása: x³-x-1=0')
plt.legend(); plt.grid(True); plt.show()

---
## 8. Többváltozós Newton-módszer

### Elmélet

**Feladat:** $F:\mathbb{R}^n\to\mathbb{R}^n$, $F(x)=0$.

**Algoritmus:** Minden lépésben megoldjuk a $F'(x^{(k)})\cdot s^{(k)} = -F(x^{(k)})$ LER-t,
majd $x^{(k+1)} = x^{(k)} + s^{(k)}$.

$$x^{(k+1)} = x^{(k)} - \bigl(F'(x^{(k)})\bigr)^{-1}F(x^{(k)})$$

ahol $F'(x)=\left(\dfrac{\partial f_i}{\partial x_j}\right)_{i,j}$ a **Jacobi-mátrix**.

### Kidolgozott példa

$$F(x) = \begin{bmatrix}f_1(x)\\f_2(x)\end{bmatrix} = \begin{bmatrix}x_1^2+x_2^2-1\\-x_1^2-x_2\end{bmatrix} = \begin{bmatrix}0\\0\end{bmatrix}$$

Geometriailag: egy fordított parabola ($x_2=-x_1^2$) és az egységkör metszéspontjai.

$$F'(x) = \begin{bmatrix}2x_1 & 2x_2\\-2x_1 & -1\end{bmatrix}$$

> **Megjegyzés:** $\det(F'(x))=0$, ha $x_1=0$ vagy $x_2=0.5$ → ott a módszer nem értelmezett.

In [ ]:
def F(x):
    return np.array([x[0]**2 + x[1]**2 - 1,
                    -x[0]**2 - x[1]])

def JF(x):
    return np.array([[2*x[0],  2*x[1]],
                    [-2*x[0], -1.0  ]])

def newton_multi(F, JF, x0, tol=1e-12, max_iter=30):
    x = np.array(x0, float); hist = [x.copy()]
    for _ in range(max_iter):
        Fx = F(x); J = JF(x)
        if abs(np.linalg.det(J)) < 1e-14:
            print('Szinguláris Jacobi!'); break
        s = np.linalg.solve(J, -Fx)
        x = x + s; hist.append(x.copy())
        if np.linalg.norm(s) < tol: break
    return x, hist

# Két különböző kezdőpontból
for x0, label in [([0.8, 0.3], 'x0=(0.8,0.3)'), ([-0.8, 0.3], 'x0=(-0.8,0.3)')]:
    sol, hist = newton_multi(F, JF, x0)
    print(f'{label}: megoldás = {sol}, F(sol) = {F(sol)}, iteráció = {len(hist)-1}')

# Ábra: parabola + kör + pályák
t = np.linspace(-1.2,1.2,300)
plt.figure(figsize=(6,6))
theta = np.linspace(0,2*np.pi,300)
plt.plot(np.cos(theta), np.sin(theta), 'b-', label='kör: x₁²+x₂²=1')
plt.plot(t, -t**2, 'g-', label='parabola: x₂=-x₁²')
for x0, col, lab in [([0.8,0.3],'r','x0=(0.8,0.3)'), ([-0.8,0.3],'m','x0=(-0.8,0.3)')]:
    _, hist = newton_multi(F, JF, x0)
    hx = np.array(hist)
    plt.plot(hx[:,0], hx[:,1], 'o-', color=col, markersize=5, label=lab)
plt.xlim(-1.5,1.5); plt.ylim(-1.5,1); plt.legend(); plt.grid(True)
plt.title('Többváltozós Newton: parabola ∩ egységkör'); plt.axis('equal'); plt.show()

---
## 9. Interpoláció – Lagrange-alak

### Elmélet

**Alapfeladat:** Adottak $x_0,\ldots,x_n$ alappontok és $y_i=f(x_i)$ értékek.
Keresünk $p_n\in P_n$-t: $p_n(x_i)=y_i$.

> **Tétel** – Az interpolációs polinom egyértelmű.

**Lagrange-alappolinomok:**
$$\ell_k(x) = \prod_{j=0,\,j\ne k}^{n}\frac{x-x_j}{x_k-x_j}, \qquad \ell_k(x_i)=\delta_{ki}$$

**Lagrange-alak:**
$$L_n(x) = \sum_{k=0}^{n} y_k\,\ell_k(x) \equiv p_n(x)$$

### Kidolgozott példa: $f(x)=\sin(x)$, alappontok: $0,\,\pi/4,\,\pi/2$

In [ ]:
def lagrange_interp(xs, ys, x):
    """Lagrange-interpoláció értéke x-ben."""
    n = len(xs)
    total = 0.0
    for k in range(n):
        lk = 1.0
        for j in range(n):
            if j != k:
                lk *= (x - xs[j]) / (xs[k] - xs[j])
        total += ys[k] * lk
    return total

xs = np.array([0, np.pi/4, np.pi/2])
ys = np.sin(xs)
print('Alappontok és értékek:')
for xi, yi in zip(xs, ys):
    print(f'  x={xi:.4f}, sin(x)={yi:.6f}')

t = np.linspace(-0.2, 1.8, 300)
p_vals = np.array([lagrange_interp(xs, ys, ti) for ti in t])

plt.figure(figsize=(8,4))
plt.plot(t, np.sin(t), 'b-', label='sin(x)')
plt.plot(t, p_vals, 'r--', label='Lagrange p₂(x)')
plt.scatter(xs, ys, s=80, color='k', zorder=5, label='alappontok')
plt.legend(); plt.grid(True)
plt.title('Lagrange-interpoláció: sin(x) közelítése 3 pontra'); plt.show()

# Ellenőrzés az alappontokon
print('Ellenőrzés az alappontokon:')
for xi, yi in zip(xs, ys):
    print(f'  p({xi:.4f}) = {lagrange_interp(xs,ys,xi):.8f}, sin = {yi:.8f}')

---
## 10. Hibaformula

> **Tétel** – Legyen $x\in\mathbb{R}$ tetszőleges, $[a,b]$ az $x_0,\ldots,x_n$ és $x$
> által kifeszített intervallum, $f\in C^{n+1}[a,b]$.
> Ekkor $\exists\,\xi_x\in[a,b]$:
> $$f(x) - p_n(x) = \frac{f^{(n+1)}(\xi_x)}{(n+1)!}\cdot\omega_n(x),$$
> ahol $\omega_n(x)=\prod_{i=0}^{n}(x-x_i)$.
> 
> **Hibabecslés:** $|f(x)-p_n(x)|\le\dfrac{M_{n+1}}{(n+1)!}|\omega_n(x)|$,
> ahol $M_{n+1}=\max_{\xi\in[a,b]}|f^{(n+1)}(\xi)|$.

### Példa: $\sin(x)$ közelítése $[0,\pi/2]$-n 2 pontra ($n=1$)

Alappontok: $x_0=0$, $x_1=\pi/2$. Ekkor $M_2=\max|\sin''(\xi)|=1$.
$$|\sin(x)-p_1(x)| \le \frac{1}{2}|x(x-\pi/2)|$$

In [ ]:
xs2 = np.array([0.0, np.pi/2])
ys2 = np.sin(xs2)

t = np.linspace(0, np.pi/2, 300)
p1 = np.array([lagrange_interp(xs2, ys2, ti) for ti in t])
actual_err = np.abs(np.sin(t) - p1)
M2 = 1.0  # max|sin''| = max|sin| = 1 a [0,pi/2]-n
err_bound = M2/2 * np.abs(t * (t - np.pi/2))

plt.figure(figsize=(9,4))
plt.subplot(1,2,1)
plt.plot(t, np.sin(t), 'b-', label='sin(x)')
plt.plot(t, p1, 'r--', label='p₁(x)')
plt.scatter(xs2, ys2, s=80, color='k', zorder=5)
plt.legend(); plt.grid(True); plt.title('sin(x) közelítése n=1')

plt.subplot(1,2,2)
plt.plot(t, actual_err, 'b-', label='tényleges hiba')
plt.plot(t, err_bound, 'r--', label='hibakorlát M₂/2·|ω|')
plt.legend(); plt.grid(True); plt.title('Hibaformula ellenőrzése')
plt.tight_layout(); plt.show()
print(f'Max tényleges hiba: {actual_err.max():.6f}')
print(f'Max hibakorlát:     {err_bound.max():.6f}')

---
## 11. Newton-alak (osztott differenciák)

### Osztott differenciák

$$f[x_i, x_{i+1}] := \frac{f(x_{i+1})-f(x_i)}{x_{i+1}-x_i}$$

$$f[x_i,x_{i+1},\ldots,x_{i+k}] := \frac{f[x_{i+1},\ldots,x_{i+k}]-f[x_i,\ldots,x_{i+k-1}]}{x_{i+k}-x_i}$$

### Newton-alak

$$N_n(x) = f[x_0] + \sum_{k=1}^{n} f[x_0,\ldots,x_k]\cdot\omega_{k-1}(x),$$
ahol $\omega_{k-1}(x)=(x-x_0)(x-x_1)\cdots(x-x_{k-1})$.

**Előny:** Új alappont hozzáadásakor csak egy tag kerül az összeghez:
$$N_{n+1}(x) = N_n(x) + f[x_0,\ldots,x_{n+1}]\cdot\omega_n(x).$$

### Kidolgozott példa: $(0,1),(1,3),(2,7),(3,13)$

(Ezek a $f(x)=x^2+x+1$ értékei.)

In [ ]:
def divided_diff_table(xs, ys):
    n = len(xs)
    T = np.zeros((n, n))
    T[:, 0] = ys
    for k in range(1, n):
        for i in range(n-k):
            T[i,k] = (T[i+1,k-1] - T[i,k-1]) / (xs[i+k] - xs[i])
    return T

def newton_interp(xs, coeffs, x):
    """Newton-alak kiértékelése Horner-módszerrel."""
    n = len(coeffs)
    result = coeffs[-1]
    for k in range(n-2, -1, -1):
        result = result * (x - xs[k]) + coeffs[k]
    return result

xs = np.array([0., 1., 2., 3.])
ys = np.array([1., 3., 7., 13.])

T = divided_diff_table(xs, ys)
coeffs = T[0, :]  # az első sor adja a Newton-együtthatókat

print('Osztott differencia táblázat:')
header = f'{'x':>6}  {'f[xi]':>8}' + ''.join(f'  {'f[x'+str(i)+'...]':>10}' for i in range(1,4))
print(header)
for i in range(len(xs)):
    row = f'{xs[i]:>6.1f}  {T[i,0]:>8.4f}'
    for k in range(1, len(xs)-i):
        row += f'  {T[i,k]:>10.4f}'
    print(row)

print(f'\nNewton-együtthatók: {coeffs}')
print('Newton-alak: N(x) =', ' + '.join(
    [f'{coeffs[0]:.1f}'] +
    [f'{c:.1f}·(x-{xs[k]:.0f})' if k==1 else f'{c:.1f}·(x-{xs[0]:.0f})···(x-{xs[k-1]:.0f})'
     for k,c in enumerate(coeffs[1:],1)]))

# Ellenőrzés
t = np.linspace(-0.5, 3.5, 300)
p_newton  = np.array([newton_interp(xs, coeffs, ti) for ti in t])
p_lagrange= np.array([lagrange_interp(xs, ys, ti) for ti in t])
f_true = t**2 + t + 1

plt.figure(figsize=(8,4))
plt.plot(t, f_true, 'b-', linewidth=2, label='f(x)=x²+x+1')
plt.plot(t, p_newton, 'r--', label='Newton-alak')
plt.plot(t, p_lagrange, 'g:', linewidth=2, label='Lagrange-alak')
plt.scatter(xs, ys, s=80, color='k', zorder=5, label='alappontok')
plt.legend(); plt.grid(True)
plt.title('Newton-alak vs Lagrange-alak (mindkettő ugyanaz a polinom)'); plt.show()
print(f'Max eltérés Newton–Lagrange: {np.max(np.abs(p_newton-p_lagrange)):.2e}')

---
## Összefoglalás

| Módszer | Konvergencia rend | Feltétel |
|---------|------------------|----------|
| Gauss–Seidel | ≥ elsőrendű | szimm. PD vagy diag. dom. |
| SOR (optimális ω) | ≥ elsőrendű (gyorsabb) | szimm. PD tridiag. |
| Richardson | elsőrendű | szimm. PD, p∈(0,2/M) |
| Intervallumfelezés | elsőrendű | f folytonos, előjelváltás |
| Fixpont-iteráció | legalább elsőrendű | kontrakció |
| Newton | **másodrendű** | f∈C², f'(x*)≠0 |
| Szelőmódszer | ≈1.618-rend | f∈C², közel x*-hoz |
| Húrmódszer | elsőrendű | f∈C², állandó előjelű f' |